# 第 7 周 - 笔记本 3：使用 QLoRA 进行微调

## 目标
使用 QLoRA 微调 Llama 3.2 进行价格预测：
1. 以4位加载基础模型
2. 配置LoRA适配器
3. 使用 SFTTrainer 进行训练
4. 将适配器保存到 HuggingFace Hub

预期结果：大约 40 美元的错误（击败 GPT-5.1！）

## 时间：2-12 小时（取决于模式和 GPU）

**重要：** 使用 GPU 在 Google Colab 上运行此程序！
- 免费 T4：约 3 小时（精简模式）
- 付费 A100：约 8-12 小时（完整模式）

## Google Colab 设置

1. 将此笔记本上传到 Google Colab
2.运行时→更改运行时类型→GPU（T4或A100）
3. 运行所有单元格

## 安装依赖项（仅限 Colab）

In [ ]:
# 取消注释并在 Google Colab 上运行
# Uncomment and run on Google Colab
# !pip install -q torch transformers datasets peft trl bitsandbytes accelerate python-dotenv huggingface-hub pydantic

## 设置环境

In [ ]:
import os
import torch
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

# 配置
# Configuration
LITE_MODE = True  # Set to False for full training
HF_TOKEN = "your_token_here"  # Replace with your HuggingFace token

# 登录 HuggingFace
# Login to HuggingFace
login(HF_TOKEN)

print("✅ Environment setup complete")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 配置

In [ ]:
# 模型和数据集
# Model and dataset
BASE_MODEL = "meta-llama/Llama-3.2-3B"
DATASET_NAME = "ed-donner/items_prompts_lite" if LITE_MODE else "ed-donner/items_prompts_full"
OUTPUT_DIR = "./llama-pricer-lite" if LITE_MODE else "./llama-pricer-full"
HUB_MODEL_ID = "your-username/llama-pricer-lite" if LITE_MODE else "your-username/llama-pricer-full"

# QLoRA 设置
# QLoRA settings
LORA_RANK = 16 if LITE_MODE else 32
LORA_ALPHA = 32 if LITE_MODE else 64
LORA_DROPOUT = 0.05

# 训练设置
# Training settings
BATCH_SIZE = 4 if LITE_MODE else 8
GRADIENT_ACCUMULATION = 4
NUM_EPOCHS = 3 if LITE_MODE else 5
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 256

print("Configuration:")
print(f"  Mode: {'LITE' if LITE_MODE else 'FULL'}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  LoRA Rank: {LORA_RANK}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")

## 加载数据集

In [ ]:
print(f"Loading dataset: {DATASET_NAME}")
dataset = load_dataset(DATASET_NAME)

train_data = dataset["train"]
val_data = dataset["val"]

print(f"✅ Dataset loaded:")
print(f"   Training: {len(train_data):,} examples")
print(f"   Validation: {len(val_data):,} examples")

# 显示示例
# Show example
print(f"\nExample training data:")
print(f"Prompt: {train_data[0]['prompt'][:100]}...")
print(f"Completion: {train_data[0]['completion']}")

## 使用 4 位量化加载基础模型

In [ ]:
# 配置 4 位量化
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {BASE_MODEL}")
print("This may take a few minutes...")

# 加载分词器
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 负载模型
# Load model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # Required for gradient checkpointing

print("✅ Model loaded in 4-bit")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 配置 LoRA 适配器

In [ ]:
# 准备 k 位训练模型
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# 配置LoRA
# Configure LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# 添加 LoRA 适配器
# Add LoRA adapters
model = get_peft_model(model, lora_config)

# 打印可训练参数
# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params

print("✅ LoRA adapters configured")
print(f"Trainable parameters: {trainable_params:,} ({trainable_pct:.2f}%)")
print(f"Total parameters: {total_params:,}")

## 配置训练

In [ ]:
# 训练论证
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    push_to_hub=False,  # We'll push manually later
)

print("✅ Training arguments configured")

## 设置训练数据集格式

In [ ]:
def formatting_func(example):
    """Format prompt-completion pairs for training"""
    return example["prompt"] + example["completion"]

print("✅ Formatting function ready")

## 创建训练器

In [ ]:
# 创建 SFT 训练器
# Create SFT Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LENGTH,
)

print("✅ Trainer created")
print(f"\nReady to train for {NUM_EPOCHS} epochs on {len(train_data):,} examples")
print(f"Estimated time: {'2-3 hours' if LITE_MODE else '8-12 hours'}")

## 火车！ 🚀

In [ ]:
print("Starting training...")
print("This will take a while. Go get coffee! ☕")
print("="*60)

# 火车
# Train
trainer.train()

print("="*60)
print("✅ Training complete!")

## 保存模型

In [ ]:
# 保存在本地
# Save locally
print(f"Saving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Model saved locally")

## 推送到 HuggingFace Hub

In [ ]:
# 推送到集线器
# Push to Hub
print(f"Pushing to HuggingFace Hub: {HUB_MODEL_ID}")
model.push_to_hub(HUB_MODEL_ID, use_auth_token=True)
tokenizer.push_to_hub(HUB_MODEL_ID, use_auth_token=True)

print("✅ Model pushed to Hub")
print(f"\n🔗 View at: https://huggingface.co/{HUB_MODEL_ID}")

## 概括

✅ 微调完成！

**我们做了什么：**
1. 以4位量化加载Llama 3.2
2.添加LoRA适配器（仅2%可训练参数）
3. 使用 SFTTrainer 对提示完成对进行训练
4. 将适配器保存到本地并保存到 HuggingFace Hub

**预期结果：**
- Lite 模式：~$65 错误
- Full mode: ~$40 error (beats GPT-5.1!)

**下一步：** `04_evaluation.ipynb` - 评估微调后的模型！